# 04 — Understanding Embedding Space

**Description:** Measure similarity with dot products and cosine similarity, find nearest neighbors, explore semantic directions, and visualize embedding spaces.
**Level:** Beginner
**Tags:** Language Models, Embeddings, Similarity, Vector Geometry, Visualization

Notebook 03 used an embedding matrix to map each token ID to a vector. But what can the position of a vector represent? In a learned embedding space, useful relationships can appear as geometric patterns: similar directions, nearby points, clusters, and offsets.

This notebook uses a tiny, deliberately constructed space. That makes the geometry easy to inspect, but the vectors are **illustrative rather than learned from real text**. By the end, you will be able to:

- compute dot products, vector lengths, and cosine similarity;
- compare one token with an entire embedding matrix;
- retrieve nearest neighbors;
- test vector directions and analogy-style relationships; and
- project higher-dimensional embeddings into two dimensions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=3, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

## 1. An embedding space is a coordinate system

Each token is a point in the same coordinate system. Real models often use hundreds or thousands of dimensions, but we will begin with two so we can draw the vectors.

Our first illustrative space places animals near related young animals and puts associated objects in the same broad regions. The axes do not have official labels—the arrangement of points is what matters.

In [ ]:
tokens_2d = ["cat", "kitten", "milk", "dog", "puppy", "leash"]
vectors_2d = np.array([
    [ 1.00,  0.70],  # cat
    [ 1.15,  0.95],  # kitten
    [ 0.75,  0.20],  # milk
    [-1.00,  0.70],  # dog
    [-1.15,  0.95],  # puppy
    [-0.75,  0.20],  # leash
])
token_to_index_2d = {token: i for i, token in enumerate(tokens_2d)}

print("matrix shape:", vectors_2d.shape)
for token, vector in zip(tokens_2d, vectors_2d):
    print(f"{token:6s} → {vector}")

### Visualize the points and vectors

A vector can be viewed either as an arrow from the origin or as the point at its tip. Both views contain the same coordinates.

In [ ]:
def plot_vectors_2d(tokens, vectors, title):
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(vectors[:, 0], vectors[:, 1], s=70, color="#4C78A8")
    for token, (x, y) in zip(tokens, vectors):
        ax.arrow(0, 0, x, y, width=0.008, alpha=0.3, color="#4C78A8",
                 length_includes_head=True)
        ax.annotate(token, (x, y), xytext=(6, 5), textcoords="offset points")
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set(title=title, xlabel="dimension 1", ylabel="dimension 2",
           xlim=(-1.5, 1.5), ylim=(-0.2, 1.3), aspect="equal")
    plt.show()

plot_vectors_2d(tokens_2d, vectors_2d, "A tiny illustrative embedding space")

## 2. Dot products measure alignment and scale

For vectors $a$ and $b$, the dot product is:

$$a \cdot b = \sum_i a_i b_i$$

A large positive dot product means the vectors point broadly in the same direction and/or have large lengths. A negative result means they point in opposing directions. A result near zero means they are close to perpendicular or one vector is very short.

In [ ]:
cat = vectors_2d[token_to_index_2d["cat"]]
kitten = vectors_2d[token_to_index_2d["kitten"]]
dog = vectors_2d[token_to_index_2d["dog"]]

def dot(a, b):
    return float(np.sum(a * b))

print("cat · kitten:", dot(cat, kitten))
print("cat · dog:   ", dot(cat, dog))
print("NumPy check: ", cat @ kitten)

### Your turn: predict the sign

Before running the next cell, predict whether each dot product will be positive or negative by looking at the plot. Then add another pair.

In [ ]:
pairs = [("dog", "puppy"), ("kitten", "leash"), ("cat", "milk")]

for left, right in pairs:
    a = vectors_2d[token_to_index_2d[left]]
    b = vectors_2d[token_to_index_2d[right]]
    print(f"{left:6s} · {right:6s} = {a @ b: .3f}")

## 3. Vector length affects the dot product

A vector's Euclidean length, or norm, is:

$$\lVert a \rVert = \sqrt{\sum_i a_i^2}$$

Doubling a vector does not change its direction, but it doubles its dot product with another vector. That can be useful when magnitude carries information, but it can obscure directional similarity.

In [ ]:
short = np.array([1.0, 1.0])
long = 5 * short
query = np.array([0.9, 1.1])

print("short length:", np.linalg.norm(short))
print("long length: ", np.linalg.norm(long))
print("query · short:", query @ short)
print("query · long: ", query @ long)

## 4. Cosine similarity compares direction

Cosine similarity divides the dot product by both lengths:

$$\cos(a,b) = \frac{a \cdot b}{\lVert a \rVert \lVert b \rVert}$$

For nonzero vectors it ranges from -1 to 1:

- `1`: same direction;
- `0`: perpendicular;
- `-1`: opposite directions.

Because scale cancels, `short` and `long` have cosine similarity 1.

In [ ]:
def cosine_similarity(a, b, eps=1e-12):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    return float((a @ b) / max(denominator, eps))

print("short vs long: ", cosine_similarity(short, long))
print("cat vs kitten: ", cosine_similarity(cat, kitten))
print("cat vs dog:    ", cosine_similarity(cat, dog))

### Normalize once, compare many times

A unit vector has length 1. After normalizing every embedding row, ordinary matrix multiplication produces all pairwise cosine similarities at once.

If `E` has shape `(vocabulary, dimensions)`, then `E @ E.T` has shape `(vocabulary, vocabulary)`. Entry `[i, j]` compares tokens `i` and `j`.

In [ ]:
def normalize_rows(matrix, eps=1e-12):
    lengths = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(lengths, eps)

normalized_2d = normalize_rows(vectors_2d)
similarity_matrix = normalized_2d @ normalized_2d.T

print("embeddings: ", vectors_2d.shape)
print("normalized: ", normalized_2d.shape)
print("similarities:", similarity_matrix.shape)
print("row lengths: ", np.linalg.norm(normalized_2d, axis=1))

## 5. Visualize every pairwise similarity

A heatmap exposes structure across the entire vocabulary. The diagonal is always 1 because each nonzero vector is perfectly similar to itself. Look for blocks of related tokens and strongly negative pairs.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(similarity_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(tokens_2d)), tokens_2d, rotation=45, ha="right")
ax.set_yticks(range(len(tokens_2d)), tokens_2d)
for i in range(len(tokens_2d)):
    for j in range(len(tokens_2d)):
        color = "white" if abs(similarity_matrix[i, j]) > 0.55 else "black"
        ax.text(j, i, f"{similarity_matrix[i, j]:.2f}", ha="center", va="center", color=color)
fig.colorbar(image, ax=ax, label="cosine similarity")
ax.set_title("Pairwise cosine similarities")
plt.tight_layout()
plt.show()

## 6. Nearest-neighbor search

To find tokens similar to a query token:

1. normalize the query vector;
2. take its dot product with every normalized embedding;
3. sort scores from largest to smallest; and
4. exclude the query token itself.

This is exact search over a tiny matrix. Large systems often use specialized approximate-nearest-neighbor indexes for speed.

In [ ]:
def nearest_neighbors(query_token, tokens, vectors, k=3):
    token_to_index = {token: i for i, token in enumerate(tokens)}
    query_index = token_to_index[query_token]
    normalized = normalize_rows(vectors)
    scores = normalized @ normalized[query_index]
    order = np.argsort(scores)[::-1]
    neighbors = [i for i in order if i != query_index][:k]
    return [(tokens[i], float(scores[i])) for i in neighbors]

for query in ["cat", "dog", "milk"]:
    print(f"{query:4s} → {nearest_neighbors(query, tokens_2d, vectors_2d)}")

### Search with a new vector

A query does not need to be an existing token. Choose coordinates in the plot, then find which token vectors point in the most similar direction.

In [ ]:
query_vector = np.array([1.0, 0.4])  # Edit me
query_unit = query_vector / np.linalg.norm(query_vector)
scores = normalized_2d @ query_unit
order = np.argsort(scores)[::-1]

[(tokens_2d[i], round(float(scores[i]), 3)) for i in order]

## 7. Semantic directions

A vector difference describes the displacement from one token to another. If similar relationships produce similar displacements, a direction can encode a reusable feature.

For example, the arrow `cat → kitten` may resemble `dog → puppy`. We can compare the two difference vectors with cosine similarity.

In [ ]:
def vector_for(token):
    return vectors_2d[token_to_index_2d[token]]

cat_to_kitten = vector_for("kitten") - vector_for("cat")
dog_to_puppy = vector_for("puppy") - vector_for("dog")

print("cat → kitten:", cat_to_kitten)
print("dog → puppy:", dog_to_puppy)
print("direction similarity:", cosine_similarity(cat_to_kitten, dog_to_puppy))

### Visualize the shared direction

Plotting both offsets at their starting points makes the parallel relationship visible. Direction comparisons ignore where an arrow begins.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(vectors_2d[:, 0], vectors_2d[:, 1], s=60, color="#4C78A8")
for token, point in zip(tokens_2d, vectors_2d):
    ax.annotate(token, point, xytext=(6, 5), textcoords="offset points")
for start, offset, color in [(vector_for("cat"), cat_to_kitten, "#E45756"),
                              (vector_for("dog"), dog_to_puppy, "#E45756")]:
    ax.arrow(*start, *offset, width=0.012, color=color, length_includes_head=True)
ax.set(title="Two similar relationship vectors", xlabel="dimension 1", ylabel="dimension 2",
       xlim=(-1.5, 1.5), ylim=(-0.1, 1.3), aspect="equal")
plt.show()

## 8. Vector analogies

A famous embedding-space experiment asks whether a relationship can be transferred:

$$\text{cat} - \text{kitten} + \text{puppy} \approx \text{dog}$$

Equivalently, start at `puppy` and subtract the shared *young-animal* direction. Then retrieve the nearest vector. Analogy arithmetic is an interesting probe, not a guarantee that every concept is represented by one clean linear direction.

In [ ]:
analogy_query = vector_for("cat") - vector_for("kitten") + vector_for("puppy")
analogy_query /= np.linalg.norm(analogy_query)
analogy_scores = normalized_2d @ analogy_query
excluded = {token_to_index_2d[token] for token in ["cat", "kitten", "puppy"]}
analogy_order = [i for i in np.argsort(analogy_scores)[::-1] if i not in excluded]

[(tokens_2d[i], round(float(analogy_scores[i]), 3)) for i in analogy_order[:3]]

## 9. Moving to more dimensions

Two dimensions are convenient for drawing but too restrictive for rich representations. The following illustrative vectors have six dimensions. We deliberately add shared patterns for animal type, youth, royalty, and gender.

Real embedding dimensions are usually not this individually interpretable. Meaning is often distributed across many coordinates.

In [ ]:
tokens_6d = ["cat", "kitten", "dog", "puppy", "man", "woman", "king", "queen"]
vectors_6d = np.array([
    [ 1.0,  0.0,  0.0,  0.0,  0.0, 0.2],  # cat
    [ 1.0,  0.0,  1.0,  0.0,  0.0, 0.2],  # kitten
    [ 0.0,  1.0,  0.0,  0.0,  0.0, 0.2],  # dog
    [ 0.0,  1.0,  1.0,  0.0,  0.0, 0.2],  # puppy
    [ 0.0,  0.0,  0.0,  1.0,  0.0, 0.2],  # man
    [ 0.0,  0.0,  0.0, -1.0,  0.0, 0.2],  # woman
    [ 0.0,  0.0,  0.0,  1.0,  1.0, 0.2],  # king
    [ 0.0,  0.0,  0.0, -1.0,  1.0, 0.2],  # queen
])
token_to_index_6d = {token: i for i, token in enumerate(tokens_6d)}

print("shape:", vectors_6d.shape)
print("king: ", vectors_6d[token_to_index_6d["king"]])

### Test two directions

Compare the *young* directions and the *royalty* directions. Identical differences have cosine similarity 1. Then modify one vector slightly and observe how robust the relationship is.

In [ ]:
def vector_6d(token):
    return vectors_6d[token_to_index_6d[token]]

young_cat = vector_6d("kitten") - vector_6d("cat")
young_dog = vector_6d("puppy") - vector_6d("dog")
royal_man = vector_6d("king") - vector_6d("man")
royal_woman = vector_6d("queen") - vector_6d("woman")

print("young directions:  ", cosine_similarity(young_cat, young_dog))
print("royalty directions:", cosine_similarity(royal_man, royal_woman))

## 10. PCA projects high-dimensional vectors to 2D

We cannot directly draw six dimensions. Principal component analysis (PCA) finds directions that capture as much variation in the points as possible, then expresses each vector using the first two directions.

Projection is lossy: distances and clusters in the plot are approximations. A visualization can suggest a pattern, but measurements should use the original vectors.

In [ ]:
def pca_2d(matrix):
    centered = matrix - matrix.mean(axis=0, keepdims=True)
    _, singular_values, components = np.linalg.svd(centered, full_matrices=False)
    projected = centered @ components[:2].T
    explained = singular_values**2 / np.sum(singular_values**2)
    return projected, explained[:2]

projected_2d, explained = pca_2d(vectors_6d)
print("input shape:     ", vectors_6d.shape)
print("projection shape:", projected_2d.shape)
print("variance captured by two components:", explained.sum().round(3))

### Visualize the projection

The axes are PCA components, not original embedding dimensions. Look for preserved relationships, but remember that overlap or separation may partly reflect the projection.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = ["#4C78A8"] * 4 + ["#F58518"] * 4
ax.scatter(projected_2d[:, 0], projected_2d[:, 1], c=colors, s=80)
for token, point in zip(tokens_6d, projected_2d):
    ax.annotate(token, point, xytext=(6, 5), textcoords="offset points")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
ax.set(title="PCA projection of six-dimensional embeddings",
       xlabel=f"PC1 ({explained[0]:.0%} of variance)",
       ylabel=f"PC2 ({explained[1]:.0%} of variance)")
plt.show()

## 11. Similarity is useful, but not meaning itself

Embedding geometry reflects the training objective and data. Similarity can mix many kinds of relationships: synonyms, shared topics, grammatical roles, frequent co-occurrence, or even dataset artifacts.

Keep these cautions in mind:

- nearest does not necessarily mean synonymous;
- cosine similarity ignores magnitude;
- one direction rarely captures a concept perfectly;
- visual projections distort high-dimensional relationships; and
- learned spaces can reproduce social and cultural biases from their data.

Treat geometry as evidence about a representation, not as a complete explanation of what a model understands.

## 12. Challenges

1. **Dot product vs cosine:** Create two vectors for which dot-product ranking and cosine-similarity ranking disagree. Explain why.
2. **Nearest neighbors:** Modify a 2D vector so `cat` becomes closer to `leash` than to `kitten`. Confirm numerically and visually.
3. **Similarity matrix:** Find the most similar pair of distinct tokens without inspecting the heatmap manually.
4. **Analogy:** Use the six-dimensional space to compute `king - man + woman`. Exclude the input tokens and retrieve its nearest neighbor.
5. **Noise:** Add small random noise to the six-dimensional vectors. How much noise can the analogy relationships tolerate?
6. **Projection:** Compare similarities measured before and after PCA. Which pair changes the most?
7. **Normalization:** Explain what happens if a zero vector is normalized without an epsilon safeguard.

In [ ]:
# Challenge workspace: solve king - man + woman.
query = vector_6d("king") - vector_6d("man") + vector_6d("woman")
normalized_6d = normalize_rows(vectors_6d)
scores = normalized_6d @ (query / np.linalg.norm(query))
excluded = {token_to_index_6d[token] for token in ["king", "man", "woman"]}
ranked = [i for i in np.argsort(scores)[::-1] if i not in excluded]

[(tokens_6d[i], round(float(scores[i]), 3)) for i in ranked[:3]]

## Takeaways

- Embeddings place tokens as vectors in a shared coordinate system.
- Dot products depend on both alignment and vector magnitude.
- Cosine similarity compares direction by normalizing away magnitude.
- Matrix multiplication can compare one query with every vocabulary vector at once.
- Difference vectors can reveal recurring directions and support analogy experiments.
- PCA makes high-dimensional structure visible, but any 2D projection loses information.
- Embedding relationships reflect training data and objectives, so interpret them cautiously.

**Next:** *05 — From Hidden State to Probabilities* will transform a model vector into logits and next-token probabilities.